In [ ]:
import torch
import os
import sys
sys.path.append('../')
from torch.utils.data import DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import StepLR
from scripts.flow import Flow
from tqdm import tqdm
from  scripts.networks.unet import UNetModel
import matplotlib.pyplot as plt
from scripts.datasets import Dataset
from torch.utils.data import DataLoader


In [ ]:

def visualize_batch(
    x_t,
    x_ref=None,
    cols=8,
    cmap='gray',
    vmin=-1,
    vmax=1,
    figsize_scale=4,
    epoch=None,
    stage='val'
):

    # -------- data --------
    x_np = x_t[:, 0].detach().cpu().numpy()
    batch_size = x_np.shape[0]

    rows = (batch_size + cols - 1) // cols
    extra_row = 1 if x_ref is not None else 0

    fig, axes = plt.subplots(
        rows + extra_row,
        cols,
        figsize=(cols * figsize_scale, (rows + extra_row) * figsize_scale)
    )
    axes = axes.flatten()

    # -------- plot images --------
    for i in range(batch_size):
        axes[i].matshow(x_np[i], cmap=cmap, vmin=vmin, vmax=vmax)
        axes[i].axis('off')

    if x_ref is not None:
        ref_np = x_ref.squeeze().detach().cpu().numpy()
        axes[batch_size].matshow(ref_np, cmap=cmap, vmin=vmin, vmax=vmax)
        axes[batch_size].set_title('Reference')
        axes[batch_size].axis('off')

    for j in range(batch_size + (1 if x_ref is not None else 0), len(axes)):
        axes[j].axis('off')

    # -------- figure-level text --------
    title_parts = []
    if stage is not None:
        title_parts.append(stage.upper())
    if epoch is not None:
        title_parts.append(f'Epoch {epoch}')

    if title_parts:
        fig.suptitle(
            ' | '.join(title_parts),
            fontsize=16,
            y=0.98
        )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
# Setting hyper-parameters for training


epochs = 300
batch_size = 4
lr_adjust_epoch = 50
batch_print_interval = 500
checkpoint_save_interval = 25
val_interval = 5
save_path = './checkpoints/cgg'
use_cfg = False
device = 'cuda'
checkpoint_path = None  # or '/path/to/miniunet_50.pth'
ODE_step = 20

def print_config(**kwargs):
    width = max(len(k) for k in kwargs) + 2
    print("\n" + "=" * 50)
    print(" Training Configuration ".center(50, "="))
    print("=" * 50)
    for k, v in kwargs.items():
        print(f"{k:<{width}}: {v}")
    print("=" * 50 + "\n")


print_config(
    epochs=epochs,
    batch_size=batch_size,
    lr_adjust_epoch=lr_adjust_epoch,
    batch_print_interval=batch_print_interval,
    checkpoint_save_interval=checkpoint_save_interval,
    save_path=save_path,
    use_cfg=use_cfg,
    device=device,
    checkpoint_path=checkpoint_path
)

In [ ]:
anno_path='../scripts/split_files'
train_anno='CGG_prior.txt'
val_anno='CGG_prior.txt'
test_anno='CGG_prior.txt'

train_anno=os.path.join(anno_path, train_anno)
val_anno=os.path.join(anno_path, val_anno)
test_anno=os.path.join(anno_path, test_anno)

training_dataset=Dataset(
        train_anno,
        preload=True,
        lines=1,
        file_size=4508,
    )
val_dataset=Dataset(
        val_anno,
        preload=True,
        lines=1,
        file_size=batch_size,
    )

train_dataloader = DataLoader(training_dataset, num_workers=1, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, num_workers=1, batch_size=batch_size, shuffle=False)   


In [ ]:

model = UNetModel(
    attention_resolutions=[32, 16, 8],   
    channel_mult=(1, 2, 4, 4, 8),
    in_channels=1,
    out_channels=1,
    use_scale_shift_norm=True,
    dropout=0.2,
    image_size=64,                  
    model_channels=128,              
    num_head_channels=64,            
    num_res_blocks=2,
    resblock_updown=True
)

model.to(device)
optimizer = AdamW(model.parameters(), lr= 1e-4)
scheduler = StepLR(optimizer, step_size=lr_adjust_epoch, gamma=0.1)

flow = Flow()




start_epoch = 0

if checkpoint_path is not None and os.path.isfile(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])

    start_epoch = checkpoint.get('epoch', 0) + 1

    print(f"Resumed from checkpoint: {checkpoint_path}")
    print(f"Start epoch set to {start_epoch}")
else:
    print("No checkpoint provided. Training from scratch.")




os.makedirs(save_path, exist_ok=True)


In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

loss_list = []

for epoch in range(start_epoch, epochs):

    # ---------- checkpoint ----------
    if (epoch % checkpoint_save_interval == 0 or epoch == epochs - 1) and epoch != start_epoch:
        print(f'Saving model {epoch} to {save_path}...')
        save_dict = dict(
            model=model.state_dict(),
            optimizer=optimizer.state_dict(),
            epoch=epoch,
            loss_list=loss_list
        )
        torch.save(save_dict, os.path.join(save_path, f'miniunet_{epoch}.pth'))

    # ---------- validation ----------
    if epoch % val_interval == 0:

        model.eval()
        with torch.no_grad():
            for _, data in enumerate(val_dataloader):

                dt = 1.0 / ODE_step
                x_1 = data[0].to(device)
                x_t = torch.randn_like(x_1)

                for j in tqdm(range(ODE_step), desc="Validation: ODE Generation", leave=False):
                    t = torch.tensor([j * dt], device=device)
                    v_pred = model(x=x_t, timesteps=t)
                    x_t = x_t + v_pred * dt

                visualize_batch(
                    x_t,
                    x_ref=None,
                    cols=8,
                    cmap='gray',
                    vmin=-1,
                    vmax=1,
                    figsize_scale=4,
                    epoch=epoch,
                    stage='Generation'
                )
                visualize_batch(
                    x_1,
                    x_ref=None,
                    cols=8,
                    cmap='gray',
                    vmin=-1,
                    vmax=1,
                    figsize_scale=4,
                    epoch=epoch,
                    stage='Labels'
                )
                break

    # ---------- training ----------
    model.train()
    batch_bar = tqdm(
    train_dataloader,
    desc=f"Epoch {epoch}/{epochs}",
    leave=False
    )

    for batch, data in enumerate(batch_bar):

        x_1 = data[0].to(device)
        t = torch.rand(x_1.size(0), device=device)

        x_t, x_0 = flow.create_flow(x_1, t)
        x_t, x_0 = x_t.to(device), x_0.to(device)

        optimizer.zero_grad()
        v_pred = model(x=x_t, timesteps=t)
        loss = flow.mse_loss(v_pred, x_1, x_0)

        loss.backward()
        optimizer.step()

        loss_list.append(loss.item())

        batch_bar.set_postfix({ 'loss': f'{loss.item():.4f}',
                                'lr': f'{optimizer.param_groups[0]["lr"]:.1e}'})
        # ---------- live plot  ----------
        if len(loss_list) % 10 == 0:   
            clear_output(wait=True)
            plt.figure(figsize=(6, 4))
            plt.plot(loss_list, linewidth=2)
            plt.xlabel('Iteration')
            plt.ylabel('Loss')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

    scheduler.step()
